# Last-Mile Delivery Optimization Using Data Science
**Week 1: Strategic Planning and Data Exploration in Logistics**

**Objective:** Explore logistics delivery records, calculate performance KPIs, visualize patterns, and build a baseline model to classify delivery status.

> **Important distinction:** The Random Forest model below is a *predictive classification model*, not a route-optimization algorithm. Actual route optimization would require locations, road-network distances/times, vehicle capacities, and operational constraints (for example, a Vehicle Routing Problem formulation).


## 1. Setup and Data Loading
Run this notebook from top to bottom in Google Colab. Upload `logistics_data.csv` when prompted, or make sure the file is available at `/content/logistics_data.csv`.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Google Colab upload (skip this cell if the CSV is already in /content)
try:
    from google.colab import files
    if not os.path.exists("/content/logistics_data.csv"):
        uploaded = files.upload()
except ImportError:
    pass

csv_path = "/content/logistics_data.csv" if os.path.exists("/content/logistics_data.csv") else "logistics_data.csv"
df = pd.read_csv(csv_path)

print(f"Initial dataset: {df.shape[0]} rows × {df.shape[1]} columns")
display(df.head())
df.info()


## 2. Data Quality Checks
Check column names, missing values, duplicate rows, and numeric ranges before analysis. This notebook removes exact duplicate rows and imputes missing values in the two numeric columns identified in the original analysis.


In [ ]:
print("Column names:", df.columns.tolist())
print("\\nMissing values before cleaning:")
display(df.isnull().sum().to_frame("Missing values"))

duplicate_count = int(df.duplicated().sum())
print(f"Exact duplicate rows: {duplicate_count}")
df = df.drop_duplicates().copy()

# Median imputation is robust to extreme values for these numeric fields.
for col in ["Distance_km", "Delivery_Time_min"]:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

print("\\nMissing values after cleaning:")
display(df.isnull().sum().to_frame("Missing values"))
print(f"Cleaned dataset: {df.shape[0]} rows × {df.shape[1]} columns")

# Confirm expected columns are present before proceeding.
required_columns = [
    "Order_ID", "Order_Date", "Distance_km", "Vehicle_Type",
    "Order_Quantity", "Traffic_Level", "Weather",
    "Delivery_Time_min", "Delivery_Status", "Delivery_Cost"
]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing expected columns: {missing_columns}")


## 3. Exploratory Data Analysis (EDA)
Start with summary statistics and delivery-status distribution. Treat the status percentages and group summaries as descriptive results for this dataset, not proof that one factor causes delays.


In [ ]:
display(df.describe(include="all").T)

status_pct = (df["Delivery_Status"].value_counts(normalize=True) * 100).round(2)
print("Delivery status distribution (%):")
display(status_pct.to_frame("Percentage"))

performance_summary = (
    df.groupby("Delivery_Status")[[
        "Delivery_Time_min", "Distance_km", "Delivery_Cost", "Order_Quantity"
    ]]
    .mean()
    .round(2)
)
print("Mean metrics grouped by delivery status:")
display(performance_summary)


In [ ]:
# Average delivery time by traffic and weather
traffic_weather = df.pivot_table(
    values="Delivery_Time_min",
    index="Traffic_Level",
    columns="Weather",
    aggfunc="mean"
).round(2)
display(traffic_weather)

# Delay percentage by vehicle type (assuming the dataset uses the label "Delayed")
delay_rate_vehicle = (
    df.assign(is_delayed=df["Delivery_Status"].eq("Delayed"))
      .groupby("Vehicle_Type")["is_delayed"]
      .mean()
      .mul(100)
      .round(2)
      .rename("Delay_Percentage")
      .reset_index()
)
display(delay_rate_vehicle)


## 4. Visualizations
These plots compare delivery status by vehicle type, average time by traffic/weather, delivery cost against distance, and numeric correlations.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.countplot(data=df, x="Vehicle_Type", hue="Delivery_Status", ax=axes[0, 0])
axes[0, 0].set_title("Delivery Status by Vehicle Type")
axes[0, 0].tick_params(axis="x", rotation=15)

sns.barplot(data=df, x="Traffic_Level", y="Delivery_Time_min", errorbar=None, ax=axes[0, 1])
axes[0, 1].set_title("Average Delivery Time by Traffic Level")

sns.scatterplot(data=df, x="Distance_km", y="Delivery_Cost", hue="Vehicle_Type", alpha=0.7, ax=axes[1, 0])
axes[1, 0].set_title("Delivery Cost vs Distance")

sns.barplot(data=df, x="Weather", y="Delivery_Time_min", errorbar=None, ax=axes[1, 1])
axes[1, 1].set_title("Average Delivery Time by Weather")
axes[1, 1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()


In [ ]:
numeric_cols = ["Distance_km", "Order_Quantity", "Delivery_Time_min", "Delivery_Cost"]
corr_matrix = df[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Logistics Metrics")
plt.tight_layout()
plt.show()


## 5. Key Performance Indicators (KPIs)
The KPIs below give a baseline operational view. Delivery delay rate is calculated from the `Delivery_Status` label. If your real business uses a promised delivery time or SLA, use that formal definition instead.


In [ ]:
total_orders = len(df)
delayed_orders = int(df["Delivery_Status"].eq("Delayed").sum())
on_time_orders = int(df["Delivery_Status"].eq("On Time").sum())

kpis = pd.DataFrame({
    "KPI": [
        "Total orders",
        "On-time delivery rate (%)",
        "Delay rate (%)",
        "Average delivery time (minutes)",
        "Average delivery cost"
    ],
    "Value": [
        total_orders,
        round(on_time_orders / total_orders * 100, 2) if total_orders else np.nan,
        round(delayed_orders / total_orders * 100, 2) if total_orders else np.nan,
        round(df["Delivery_Time_min"].mean(), 2),
        round(df["Delivery_Cost"].mean(), 2)
    ]
})
display(kpis)


## 6. Predictive Modeling: Delivery Status Classification
We will predict `Delivery_Status` using information that could be known before delivery: distance, vehicle type, order quantity, traffic level, and weather.

**Leakage prevention:** `Delivery_Time_min` and `Delivery_Cost` are excluded from the predictors here. These may be known only after or during fulfillment, and using them can make an evaluation unrealistically optimistic. `Order_ID` is an identifier and `Order_Date` is omitted from this baseline model.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.impute import SimpleImputer

feature_cols = [
    "Distance_km", "Vehicle_Type", "Order_Quantity",
    "Traffic_Level", "Weather"
]
target_col = "Delivery_Status"

model_df = df[feature_cols + [target_col]].dropna(subset=[target_col]).copy()
X = model_df[feature_cols]
y = model_df[target_col]

numeric_features = ["Distance_km", "Order_Quantity"]
categorical_features = ["Vehicle_Type", "Traffic_Level", "Weather"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ("categorical", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight="balanced"
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix — Delivery Status")
plt.tight_layout()
plt.show()


## 7. Interpreting Results
Use the printed classification report and confusion matrix to discuss model performance. Accuracy alone may hide poor performance on a less common class, so also review precision, recall, and F1-score for both classes.

This baseline model identifies patterns in the provided dataset; it does not guarantee real-world performance. Validate it on fresh, representative delivery data before operational use.


In [ ]:
# Feature importance from the fitted Random Forest after preprocessing
preprocessor_fitted = model.named_steps["preprocessor"]
feature_names = preprocessor_fitted.get_feature_names_out()
importances = model.named_steps["classifier"].feature_importances_

importance_df = (
    pd.DataFrame({"Feature": feature_names, "Importance": importances})
      .sort_values("Importance", ascending=False)
      .head(15)
)

display(importance_df)

plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Top Feature Importances — Random Forest")
plt.tight_layout()
plt.show()


## 8. Limitations and Future Work
- This notebook uses a dataset with a limited set of operational variables; results may not generalize to all cities or seasons.
- Association in EDA does not establish causation.
- Delivery status labels and data quality should be checked with the logistics team.
- A true **route optimization** stage is separate from this classifier. It would need pickup/drop-off coordinates or a road-distance matrix, vehicle capacity, delivery time windows, and business constraints. A Vehicle Routing Problem (VRP) solver or optimization library could then be evaluated.
- Future work could compare classification algorithms, validate on time-based holdout data, predict delivery time using regression, and test route/resource allocation scenarios.


## 9. Conclusion
This workflow cleans the data, summarizes delivery performance, calculates core KPIs, explores relationships among traffic, weather, vehicle type, distance, time, and cost, and builds a baseline Random Forest classifier for delivery status. The next practical step is to validate findings with real operational data and develop a separate route-optimization formulation if route planning is in scope.

**Reproducibility:** Run all cells from top to bottom before submission so that tables, plots, and evaluation metrics reflect the dataset currently being used.
